In [15]:
pip install python-dotenv requests pandas numpy scikit-learn joblib tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
from pathlib import Path
from dotenv import dotenv_values

env_candidates = [
    Path.cwd() / ".env",
    Path("/content/Time-Series-Forecasting/.env"),
    Path("/content/.env"),
]
env_path = next((path for path in env_candidates if path.is_file()), None)

if env_path is None:
    print("No .env file found. Upload .env to the notebook runtime to load the API token.")
else:
    print("Using:", env_path.resolve())
    values = dotenv_values(env_path)
    print("Variables found:", list(values.keys()))

Using: C:\Users\LENOVO\OneDrive\Desktop\alu-machine_learning\Time-Series-Forecasting\.env
Variables found: ['DATAVERSE_API_TOKEN']


In [17]:
from dotenv import load_dotenv
import os

if env_path is not None:
    load_dotenv(dotenv_path=env_path)

API_TOKEN = os.getenv("DATAVERSE_API_TOKEN")

if API_TOKEN:
    print("API token loaded successfully.")
else:
    print("API token was not found. Upload .env and rerun this cell.")

API token loaded successfully.


In [18]:
# configuring Harvard Dataverse
import requests

SERVER = "https://dataverse.harvard.edu"

headers = {
    "X-Dataverse-key": API_TOKEN
}

DOI = "doi:10.7910/DVN/EGZHFV"

In [19]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API_TOKEN = os.getenv("DATAVERSE_API_TOKEN")

print("Token exists:", API_TOKEN is not None)
print("Token length:", len(API_TOKEN) if API_TOKEN else 0)

Token exists: True
Token length: 36


In [20]:
import requests
import os
from dotenv import load_dotenv

# Load .env
load_dotenv()

# Dataverse server
SERVER = "https://dataverse.harvard.edu"

# Your dataset DOI
DOI = "doi:10.7910/DVN/EGZHFV"

# API token from .env
API_TOKEN = os.getenv("DATAVERSE_API_TOKEN")

headers = {"X-Dataverse-key": API_TOKEN}

In [21]:
metadata_url = f"{SERVER}/api/datasets/:persistentId/"

params = {
    "persistentId": DOI
}

response = requests.get(
    metadata_url,
    params=params,
    headers=headers
)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("Dataverse API connection successful!")
else:
    print(response.text[:2000])

Status code: 200
Dataverse API connection successful!


Discover the dataset files

In [22]:
metadata_url = f"{SERVER}/api/datasets/:persistentId/"

response = requests.get(
    metadata_url,
    params={"persistentId": DOI},
    headers=headers
)

metadata = response.json()

files = metadata["data"]["latestVersion"]["files"]

print("Total files:", len(files))

for i, f in enumerate(files[:15]):
    print(i, f["dataFile"]["filename"])

Total files: 62
0 sms-call-internet-mi-2013-11-01.txt
1 sms-call-internet-mi-2013-11-02.txt
2 sms-call-internet-mi-2013-11-03.txt
3 sms-call-internet-mi-2013-11-04.txt
4 sms-call-internet-mi-2013-11-05.txt
5 sms-call-internet-mi-2013-11-06.txt
6 sms-call-internet-mi-2013-11-07.txt
7 sms-call-internet-mi-2013-11-08.txt
8 sms-call-internet-mi-2013-11-09.txt
9 sms-call-internet-mi-2013-11-10.txt
10 sms-call-internet-mi-2013-11-11.txt
11 sms-call-internet-mi-2013-11-12.txt
12 sms-call-internet-mi-2013-11-13.txt
13 sms-call-internet-mi-2013-11-14.txt
14 sms-call-internet-mi-2013-11-15.txt


Download one day first

In [23]:
import os

os.makedirs("data", exist_ok=True)

file_info = files[0]

file_id = file_info["dataFile"]["id"]
filename = file_info["dataFile"]["filename"]

download_url = f"{SERVER}/api/access/datafile/{file_id}"

response = requests.get(download_url, headers=headers)

with open(f"data/{filename}", "wb") as f:
    f.write(response.content)

print("Downloaded:", filename)

Downloaded: sms-call-internet-mi-2013-11-01.txt


In [24]:
file_info = files[0]

file_id = file_info["dataFile"]["id"]
filename = file_info["dataFile"]["filename"]

download_url = f"{SERVER}/api/access/datafile/{file_id}"

response = requests.get(download_url, headers=headers)

print("Status:", response.status_code)
print(response.text[:300])

Status: 400
{"status":"ERROR","message":"You may not download this file without the required Guestbook response for guestbookID 96."}


Inspect the file

In [25]:
import pandas as pd

sample = pd.read_csv(
    f"data/{filename}",
    sep="\t",
    nrows=5,
    header=None
)

sample

,0
0,"{""status"":""ERROR"",""message"":""You may not downl..."


In [26]:
sample.columns = [
    "SquareID",
    "TimeInterval",
    "CountryCode",
    "SMS_in",
    "SMS_out",
    "Call_in",
    "Call_out",
    "Internet"
]

sample

ValueError: Length mismatch: Expected axis has 1 elements, new values have 8 elements

Measure memory usage (Task 1)

In [ ]:
df = pd.read_csv(
    f"data/{filename}",
    sep="\t",
    header=None,
    names=[
        "SquareID","TimeInterval","CountryCode",
        "SMS_in","SMS_out","Call_in","Call_out","Internet"
    ]
)

df.memory_usage(deep=True).sum()/1024**2

Optimize memory

In [ ]:
df["SquareID"] = df["SquareID"].astype("int16")
df["CountryCode"] = df["CountryCode"].astype("int16")

for col in ["SMS_in","SMS_out","Call_in","Call_out","Internet"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

measure again

In [ ]:
optimized_memory = df.memory_usage(deep=True).sum()/1024**2

print(optimized_memory)

In [ ]:
Then calculate the improvement

In [ ]:
before = 42   # replace with your value
after = optimized_memory

improvement = (before-after)/before*100

print(f"Memory reduced by {improvement:.1f}%")

Download all files

In [ ]:
import os

os.makedirs("data", exist_ok=True)

for f in files:
    file_id = f["dataFile"]["id"]
    filename = f["dataFile"]["filename"]

    path = f"data/{filename}"

    if os.path.exists(path):
        continue

    url = f"{SERVER}/api/access/datafile/{file_id}"

    r = requests.get(url, headers=headers)

    with open(path, "wb") as out:
        out.write(r.content)

    print("Downloaded", filename)

Combine safely

Instead of loading everything at once, process file by file.

In [ ]:
from pathlib import Path

total_internet = {}

for path in sorted(Path("data").glob("*.txt")):

    chunk = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=[
            "SquareID","TimeInterval","CountryCode",
            "SMS_in","SMS_out","Call_in","Call_out","Internet"
        ]
    )

    sums = chunk.groupby("SquareID")["Internet"].sum()

    for square, value in sums.items():
        total_internet[square] = total_internet.get(square,0)+value